In [ ]:
import json
from openai import OpenAI

client = OpenAI(
    api_key="sk-XXX",  # <--- 请在此处填入您的 API KEY
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)


def search_flights(origin: str, destination: str, date: str):
    """模拟查询航班信息"""
    print(f"🔧 [执行工具]: 查询从 {origin} 到 {destination}，日期 {date} 的航班...")
    # 模拟外部 API 返回结果
    return {
        "flights": [
            {"flight_no": "CZ3101", "depart": "07:00", "arrive": "09:20", "price": 550},
            {"flight_no": "MU5103", "depart": "10:30", "arrive": "12:45", "price": 680},
        ]
    }


def search_hotels(city: str, date: str):
    """模拟查询酒店信息"""
    print(f"🔧 [执行工具]: 查询 {city}，日期 {date} 的酒店推荐...")
    return {
        "hotels": [
            {"hotel_name": "上海外滩大酒店", "location": "外滩", "price": 800},
            {"hotel_name": "上海静安香格里拉", "location": "静安寺", "price": 1200},
        ]
    }


# 构建一个函数映射字典，方便通过名字调用
available_functions = {
    "search_flights": search_flights,
    "search_hotels": search_hotels,
}


# ==========================================
# 3. 定义工具 Schema 描述给 LLM (JSON Schema 标准)
# ==========================================
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_flights",
            "description": "查找指定出发地、目的地和日期的航班信息。",
            "parameters": {
                "type": "object",
                "properties": {
                    "origin": {
                        "type": "string",
                        "description": "出发城市或机场（三字码）",
                    },
                    "destination": {
                        "type": "string",
                        "description": "抵达城市或机场（三字码）",
                    },
                    "date": {
                        "type": "string",
                        "description": "出发日期，格式 YYYY-MM-DD",
                    },
                },
                "required": ["origin", "destination", "date"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search_hotels",
            "description": "搜索指定城市在指定日期的酒店选项。",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "目标城市名称"},
                    "date": {
                        "type": "string",
                        "description": "入住日期，格式 YYYY-MM-DD",
                    },
                },
                "required": ["city", "date"],
            },
        },
    },
]


def run_travel_assistant():
    # 用户初始问题
    user_prompt = "请帮我查找7月20日北京飞上海的航班，并推荐上海市中心的酒店。"
    print(f"🧑‍💻 [用户]: {user_prompt}\n")

    messages = [{"role": "user", "content": user_prompt}]

    # 第一轮调用：LLM 决定是否需要调用工具
    print("🤖 [模型思考中...]")
    response = client.chat.completions.create(
        model="qwen-plus",  # 可根据您的权限修改为 qwen-turbo 或 qwen-max
        messages=messages,
        tools=tools,
        tool_choice="auto",
    )

    response_message = response.choices[0].message
    # 为什么叫 choices（选择）而且是个列表？ 因为 API 的底层设计允许您一次性要求模型生成多个不同
    # 的回答（比如设置参数 n=3，让模型给出 3 个草稿供您挑选）。

    # 但在绝大多数情况下（包括我们的代码中），我们没有设置 n 参数，默认只让模型生成
    # 1 个回答。所以这个列表里其实只有 1 个元素。
    tool_calls = response_message.tool_calls

    if tool_calls:
        print(f"💡 [模型请求]: 检测到需要调用 {len(tool_calls)} 个工具。\n")
        messages.append(response_message)

        for tool_call in tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)

            # 根据名称找到本地映射的函数
            function_to_call = available_functions.get(function_name)

            if function_to_call:
                function_response = function_to_call(**function_args)

                # 将执行结果作为 "tool" 角色追加到消息列表中 (主流标准做法，非 user 角色)
                messages.append(
                    {
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "name": function_name,
                        "content": json.dumps(function_response, ensure_ascii=False),
                    }
                )

        # 第二轮调用：带着外部数据，让 LLM 生成最终的自然语言回答
        print("\n🤖 [数据已获取，模型生成最终回答...]")
        second_response = client.chat.completions.create(
            model="qwen-plus", messages=messages
        )

        final_answer = second_response.choices[0].message.content
        print(f"\n✨ [最终回答]:\n{final_answer}")
    else:
        # 如果不需要工具，直接输出回答
        print(f"\n✨ [最终回答]:\n{response_message.content}")


In [3]:
run_travel_assistant()


🧑‍💻 [用户]: 请帮我查找7月20日北京飞上海的航班，并推荐上海市中心的酒店。

🤖 [模型思考中...]
💡 [模型请求]: 检测到需要调用 2 个工具。

🔧 [执行工具]: 查询从 PEK 到 SHA，日期 2023-07-20 的航班...
🔧 [执行工具]: 查询 上海，日期 2023-07-20 的酒店推荐...

🤖 [数据已获取，模型生成最终回答...]

✨ [最终回答]:
根据您的需求，我为您查询了2023年7月20日北京（首都国际机场PEK）飞往上海（虹桥机场SHA）的航班，并筛选了两家位于上海市中心、交通便利、口碑优良的高性价比酒店推荐：

✅ **优选航班推荐（7月20日）**  
- **中国南方航空 CZ3101**  
  🕒 起飞：07:00（PEK）｜抵达：09:20（SHA）  
  💰 含税票价约 ¥550（经济舱，实时比价中低价）  
  ✅ 优势：早班机，抵达后时间充裕；准点率高，航程约2h20m  

- **东方航空 MU5103**  
  🕒 起飞：10:30（PEK）｜抵达：12:45（SHA）  
  💰 含税票价约 ¥680  
  ✅ 优势：午间抵达，衔接市区交通方便；东航上海主基地，服务成熟  

📌 温馨提示：  
• 建议提前2小时抵达机场办理值机；  
• SHA（虹桥）距市中心（如人民广场/外滩）约20分钟地铁（2号线直达），比浦东机场（PVG）更便捷；  
• 可通过航司App或“民航官方APP”（如航旅纵横）实时查动态、选座、值机。

🏨 **上海市中心优质酒店推荐**（均步行可达地铁，近核心商圈/景点）  
1. **上海外滩大酒店**  
   📍 位置：黄浦区中山东一路27号｜正对外滩万国建筑群，步行至南京东路步行街约5分钟  
   💰 参考价：¥800/晚（含早，7月旺季价格，大床房）  
   ✅ 亮点：江景视野佳、历史底蕴深厚、地铁10/2号线「南京东路站」步行3分钟  

2. **上海静安香格里拉大酒店**  
   📍 位置：静安区延安中路1218号｜毗邻静安寺、嘉里中心，地铁2/7号线「静安寺站」上盖  
   💰 参考价：¥1200/晚（含早，行政大床房）  
   ✅ 亮点：高端服务与设施、餐饮选择丰富（夏宫中餐厅知名）、购物/文化体验集中  

💡 小贴士：  

```
                messages.append(
                    {
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "name": function_name,
                        "content": json.dumps(function_response, ensure_ascii=False),
                    }
                )
```
# "tool_call_id": tool_call.id, ID是怎么来的  ？？

原因： 唯一标识符，涉及并行工具调用。因此名称

这是一个非常细节且极其重要的问题！

简单直接地回答：**这个 `id` 是由大模型所在的云端服务器（比如 OpenAI 或阿里云的后台）自动生成的，并在它决定调用工具的那一瞬间，连同工具名称和参数一起返回给您的。**

为了让您彻底明白它的来龙去脉和核心作用，我们把整个过程拆解来看：

### 1. 它是从哪里“生”出来的？

在前面的代码中，当您第一次向模型发起提问时：
```python
response = client.chat.completions.create(...)
```
大模型思考后，如果在返回值里包含了 `tool_calls`，如果我们把底层的那个 `tool_call` 对象打印出来，它长这样：

```json
{
  "id": "call_abc123xyz890",  // <--- 就是这里！服务器自动生成的唯一编号
  "type": "function",
  "function": {
    "name": "search_flights",
    "arguments": "{\"origin\": \"北京\", \"destination\": \"上海\", \"date\": \"2024-07-20\"}"
  }
}
```
所以，您代码里的 `tool_call.id`，就是直接提取了服务器返回的这段像身份证号一样的字符串（比如 `"call_abc123xyz890"`）。

---

### 2. 为什么一定要有这个 ID？（核心原因：并行调用）

您可能会想：“反正我已经通过 `name: search_flights` 告诉模型我执行的是哪个工具了，为什么还要多此一举传个 ID？”

这是因为现在的主流大模型都支持一个强大的功能：**并行工具调用（Parallel Tool Calling）**。

假设用户问：**“请帮我查一下北京、上海和广州明天的天气。”**
此时，大模型为了节省时间，**不会**一次只查一个，而是会**同时发起 3 个**工具调用请求。

此时，服务器返回的 `tool_calls` 列表会包含 3 个对象：
1. `id`: "call_001", `name`: "get_weather", `args`: "北京"
2. `id`: "call_002", `name`: "get_weather", `args`: "上海"
3. `id`: "call_003", `name`: "get_weather", `args`: "广州"

接下来，您的本地代码（调度器）去分别执行这 3 次查询，得到了 3 个结果（比如 25度、28度、30度）。

**如果不带 ID 塞回去，会发生什么？**
您把 3 个 25度、28度、30度 作为 `role: tool` 塞回给大模型。大模型拿到后会彻底懵圈：“这三个结果到底哪个是北京的？哪个是广州的？”

**带上 ID 塞回去：**
您告诉大模型：
* “报告！`call_001` 的执行结果是 25 度。”
* “报告！`call_002` 的执行结果是 28 度。”
大模型内部的机制就可以根据 `id` 严丝合缝地将结果对号入座，绝不会把北京的天气报给广州。

---

### 💡 总结：一个通俗的比喻

您可以把大模型想象成一家餐厅的**服务员**，您的本地代码是**后厨**。

1. 服务员（大模型）一次性在前厅接了三桌的菜单，走到后厨把单子递给您（发起 3 个 `tool_calls`）。
2. 为了防止上错菜，服务员在每一张单子上都夹了一个**唯一的排队小票号码**（这就是 `tool_call_id`）。
3. 后厨（您的代码）把菜炒好后，必须**把对应的小票号码和菜一起端给服务员**（塞回 `messages` 历史记录）。
4. 服务员一看号码，就知道哪盘菜该端给哪一桌了。

所以，把 `tool_call_id` 传回去，是完成工具调用闭环、防止数据错乱的强制性“握手协议”。